In [1]:
import mofr

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, time, timedelta
from SynthSpread.spreadviewer_class import SpreadSingle, SpreadViewerData, norm_coeff
#from Database.TPData import TPData, TPDataDa, TPDataAssembly

from Strategies.LeadLagRegression_strategy.backtest_class import BacktestLL
from Strategies.LeadLagRegression_strategy.strategy_class import StrategyLL, VolumeClass
from support_functions import calculate_MACD, calculate_regression_model_price, calc_vol_intensity_index, calculate_regression_model_price_new
tol=(1e-1)/2

import seaborn as sns

from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, Legend
from bokeh.io import output_notebook
from scipy.stats import gaussian_kde

In [3]:
import seaborn as sns
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from xgboost import XGBClassifier, plot_tree
import xgboost as xgb
import shap
from sklearn.model_selection import train_test_split
from sklearn import tree
from sklearn.tree import export_text
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [4]:
%load_ext autoreload
%autoreload 2

In [5]:
import warnings
warnings.filterwarnings('ignore')

In [6]:
burnout_period=60
stop_profit=0.3
makeagg_ratio=0.6
trail_stop=0

take_profit=2
stop_loss=-1

br_fee = 0.0175
closing_mode='Martinovo_zatvaranie'

In [7]:
##########################
param_dict = {}
param_dict['t_end'] = time(17)
param_dict['take_profit'] = take_profit
param_dict['stop_loss'] = stop_loss
param_dict['ba_max']=0.3

param_dict['burnout_period']=burnout_period
param_dict['stop_profit']=stop_profit
param_dict['makeagg_ratio']=makeagg_ratio
param_dict['trail_stop']=trail_stop


param_dict['br_fee'] = br_fee
##########################


strategy_class = StrategyLL(
                            strategy='LL',
                            market='de',
                            instrument='m2',
                            is_overnight=False,
                            closing_mode=closing_mode)

strategy_class.param_dict=param_dict

# Data preparation

In [8]:
data_lead =pd.concat([pd.read_csv(r's:\Algo\Files\andrej\Data\int_data_lead_dem1_jan_feb.csv',
                    parse_dates=['datetime']).reset_index(), pd.read_csv(r's:\Algo\Files\andrej\Data\int_data_lead_dem1_mar_apr.csv',
                    parse_dates=['datetime']).reset_index(), pd.read_csv(r's:\Algo\Files\andrej\Data\int_data_lead_dem1_may_june.csv',
                    parse_dates=['datetime']).reset_index(), pd.read_csv(r's:\Algo\Files\andrej\Data\int_data_lead_dem1_july.csv',
                    parse_dates=['datetime']).reset_index(), pd.read_csv(r's:\Algo\Files\andrej\Data\int_data_lead_dem1_aug.csv',
                    parse_dates=['datetime']).reset_index()])

data_lag=pd.concat([pd.read_csv(r's:\Algo\Files\andrej\Data\int_data_lag_dem2_jan_feb.csv',
                    parse_dates=['datetime']).reset_index(), pd.read_csv(r's:\Algo\Files\andrej\Data\int_data_lag_dem2_mar_apr.csv',
                    parse_dates=['datetime']).reset_index(), pd.read_csv(r's:\Algo\Files\andrej\Data\int_data_lag_dem2_may_june.csv',
                    parse_dates=['datetime']).reset_index(), pd.read_csv(r's:\Algo\Files\andrej\Data\int_data_lag_dem2_july.csv',
                    parse_dates=['datetime']).reset_index(), pd.read_csv(r's:\Algo\Files\andrej\Data\int_data_lag_dem2_aug.csv',
                    parse_dates=['datetime']).reset_index()])

data_lead2=pd.read_csv(r's:\Algo\Files\andrej\Data\int_data_lead_ttfda1_jan_aug.csv', parse_dates=['datetime']).reset_index()

In [9]:
data_lead['datetime']=pd.to_datetime(data_lead['datetime'], format='mixed')
data_lag['datetime']=pd.to_datetime(data_lag['datetime'], format='mixed')
data_lead2['datetime']=pd.to_datetime(data_lead2['datetime'], format='mixed')

In [10]:
data_lag['time_diff']=data_lag['datetime'].diff().dt.total_seconds().fillna(0)
data_lead['time_diff']=data_lead['datetime'].diff().dt.total_seconds().fillna(0)
data_lead2['time_diff']=data_lead2['datetime'].diff().dt.total_seconds().fillna(0)

In [11]:
data_lead=data_lead[data_lead['datetime'].apply(lambda x: x.hour>8 and  x.hour<18)]
data_lag=data_lag[data_lag['datetime'].apply(lambda x: x.hour>8 and  x.hour<18)]
data_lead2=data_lead2[data_lead2['datetime'].apply(lambda x: x.hour>8 and  x.hour<18)]

In [12]:
data_lag['date']=data_lag['datetime'].apply(lambda x: x.date())

## Predictor calculation

In [13]:
from Strategies.LeadLagEns.lead_lag_ensamble import LMOnline, LMOffline, DataClass
 
n_t = 15
d_t = 1

# Data properties
mkt = 'dem2'
mkt_l = ['dem1']
tau = 10
tau_ema = 10
n_ema = 16
 
diff_bool = True
vol_bool = False
scale_bool = True
isEma = False
 
tick_val = 200

km_bool = False
 
intercept = False
lambda1=.2
lambda2=.0
alpha=17.8
beta=2.9
 
adaptive_l1 = False
norm_g = True
 
model_class = LMOnline(1, intercept, lambda1, lambda2, alpha, beta,
                        adaptive_l1, norm_g)
model_class = LMOffline(intercept)
# Data preparation
data_class = DataClass(mkt, tau, tau_ema, diff_bool, vol_bool, scale_bool,
                       isEma, n_ema)

dict_lead={'dem1': data_lead}

In [14]:
data_lag['timestamp']=data_lag['datetime']
data_lead['timestamp']=data_lead['datetime']
data_lead2['timestamp']=data_lead2['datetime']


data_lag=calculate_MACD(data_lag, 6,14)
data_lag=calculate_MACD(data_lag, 12,26)
data_lag=calculate_MACD(data_lag, 18,38)

data_lead=calculate_regression_model_price(data_lead, data_lag, 10, 10, 3)
data_lead2=calculate_regression_model_price(data_lead2, data_lag, 10, 10, 3)


# a=calculate_regression_model_price_new(dict_lead, data_lag, model_class, data_class,n_t, d_t, km_bool, vol_bool=False).reset_index()
# a['timestamp']=a['index']
# data_lead=data_lead.merge(a[(a['fair_price'].isnull()==False)][['timestamp', 'fair_price', 'ret']].drop_duplicates(), how='left')

data_lag['tag']='lag'
data_lead['tag']='lead1'
data_lead2['tag']='lead2'


# data_lag= pd.concat([data_lag, data_lead[['datetime', 'timestamp', 'tag', 'trd_price','fair_price','ret', 'lag_price_predicted', 'lag_price_predicted_datetime', 'lag_price_tick', 'coef1', 'coef2', 'lead_price_tick']].reset_index(drop=True)
#                     , data_lead2[['datetime', 'timestamp', 'tag', 'trd_price','fair_price','ret', 'lag_price_predicted', 'lag_price_predicted_datetime', 'lag_price_tick', 'coef1', 'coef2', 'lead_price_tick']].reset_index(drop=True)]).reset_index(
#     drop=True).sort_values('datetime').reset_index()

data_lag= pd.concat([data_lag, data_lead[['datetime', 'timestamp', 'trd_price','tag', 'lag_price_predicted', 'lag_price_predicted_datetime', 'lag_price_tick', 'coef1', 'coef2', 'lead_price_tick']].reset_index(drop=True)
                    , data_lead2[['datetime', 'timestamp', 'trd_price', 'tag', 'lag_price_predicted', 'lag_price_predicted_datetime', 'lag_price_tick', 'coef1', 'coef2', 'lead_price_tick']].reset_index(drop=True)]).reset_index(
    drop=True).sort_values('datetime').reset_index()

del data_lag['level_0'], data_lag['index']

# Convert the timestamp to date and set it as a separate column if not already done
data_lag['date'] = data_lag['datetime'].dt.date


# data_lag=calc_vol_intensity_index(data_lag, 3, 'lead')
# data_lag=calc_vol_intensity_index(data_lag, 5, 'lead')
# data_lag=calc_vol_intensity_index(data_lag, 10, 'lead')
data_lag=calc_vol_intensity_index(data_lag, 3, 'lag')
data_lag=calc_vol_intensity_index(data_lag, 5, 'lag')
data_lag=calc_vol_intensity_index(data_lag, 10, 'lag')



# Forward fill within each day
data_lag['bid_price'] = data_lag.groupby('date')['bid_price'].ffill()
data_lag['ask_price'] = data_lag.groupby('date')['ask_price'].ffill()
data_lag['mid_price'] = data_lag.groupby('date')['mid_price'].ffill()
data_lag['MACD_6_14'] = data_lag.groupby('date')['MACD_6_14'].ffill()
data_lag['MACD_12_26'] = data_lag.groupby('date')['MACD_12_26'].ffill()
data_lag['MACD_18_38'] = data_lag.groupby('date')['MACD_18_38'].ffill()


data_lag=data_lag.dropna(subset=['bid_price', 'ask_price', 'mid_price'])

In [15]:
#data_lag[['fair_price', 'lag_price_predicted']].corr()

In [16]:
#data_lag[['fair_price', 'lag_price_predicted']].hist()

## Target calculation

In [17]:
def calculate_close_slot_long(open_price, open_time, row):
    
        if (row['ask_price']-row['bid_price']>param_dict['ba_max']) or row['tag']!='lag':
            return (0,0)
    
        strategy_class = StrategyLL(
                            strategy='LL',
                            market='de',
                            instrument='m2',
                            is_overnight=False,
                            closing_mode=closing_mode)

        strategy_class.param_dict=param_dict
        
        strategy_class.timestamp_ = row['datetime']
        strategy_class.mid_p = row['mid_price']
        strategy_class.bid_p = row['bid_price']
        strategy_class.ask_p = row['ask_price']
        strategy_class.trd_p = row['trd_price']
        strategy_class.trd_s = row['trd_side']
        strategy_class.time_diff=row['time_diff']
        strategy_class.tag=row['tag']
        
        #strategy_class.MACD = row['MACD']
        #strategy_class.lag_price_predicted_datetime = row['lag_price_predicted_datetime']
        #strategy_class.lag_price_tick = row['lag_price_tick']
        #strategy_class.lead_price_tick = row['lead_price_tick']
        
        #strategy_class.coef1 = row['coef1']
        #strategy_class.coef2 = row['coef2']

        strategy_class.position_dict['open_price']=open_price
        strategy_class.position_dict['volume']=1
        strategy_class.position_dict['open_time']=open_time
        
        strategy_class._position=1
        
        return strategy_class.calculate_close_slot()
    
def calculate_close_slot_short(open_price, open_time, row):

        if row['ask_price']-row['bid_price']>param_dict['ba_max'] or row['tag']!='lag':
            return (0,0)
        
        strategy_class = StrategyLL(
                            strategy='LL',
                            market='de',
                            instrument='m2',
                            is_overnight=False,
                            closing_mode=closing_mode)

        strategy_class.param_dict=param_dict
        
        strategy_class.timestamp_ = row['datetime']
        strategy_class.mid_p = row['mid_price']
        strategy_class.bid_p = row['bid_price']
        strategy_class.ask_p = row['ask_price']
        strategy_class.trd_p = row['trd_price']
        strategy_class.trd_s = row['trd_side']
        strategy_class.time_diff=row['time_diff']
        strategy_class.tag=row['tag']
        
        
#         strategy_class.MACD = row['MACD']
#         strategy_class.lag_price_predicted_datetime = row['lag_price_predicted_datetime']
#         strategy_class.lag_price_tick = row['lag_price_tick']
#         strategy_class.lead_price_tick = row['lead_price_tick']
        
#         strategy_class.coef1 = row['coef1']
#         strategy_class.coef2 = row['coef2']
        
        strategy_class.position_dict['open_price']=open_price
        strategy_class.position_dict['volume']=-1
        strategy_class.position_dict['open_time']=open_time
        
        strategy_class._position=-1
        
        return strategy_class.calculate_close_slot()

    
def calculate_profit_long(row, df):
    
    if row['ask_price']-row['bid_price']>param_dict['ba_max']:
        return None
    
    open_price=row['ask_price']
    open_time=row['datetime']
    
    filtered_df=df[(df['date']==row['date'])&(df['datetime']>row['datetime'])]
    
    for index, row in filtered_df.iterrows():
        price,vol = calculate_close_slot_long(open_price, open_time, row)
        
        if price!=0 and vol!=0:
            return price-open_price-2*param_dict['br_fee']
    
    return 0

def calculate_profit_short(row, df):
    
    if row['ask_price']-row['bid_price']>param_dict['ba_max']:
        return None
    
    open_price=row['bid_price']
    open_time=row['datetime']
    
    filtered_df=df[(df['date']==row['date'])&(df['datetime']>row['datetime'])]
    
    for index, row in filtered_df.iterrows():
        price,vol = calculate_close_slot_short(open_price, open_time, row)
        
        if price!=0 and vol!=0:
            return open_price-price-2*param_dict['br_fee']
    
    return 0

In [18]:
df_filtered=data_lag[data_lag['tag']!='lag']

In [ ]:
for date in sorted(list(set(df_filtered.date))):
    print(date)
    a=data_lag[data_lag['date']==date]
    df_filtered.loc[df_filtered['date']==date, 'profit_long']=df_filtered.loc[df_filtered['date']==date][['datetime', 'date', 'bid_price', 'ask_price']].apply(lambda row: calculate_profit_long(row, a), axis=1)
    df_filtered.loc[df_filtered['date']==date, 'profit_short']=df_filtered.loc[df_filtered['date']==date][['datetime', 'date', 'bid_price', 'ask_price']].apply(lambda row: calculate_profit_short(row, a), axis=1)

2024-01-02
2024-01-03
2024-01-04
2024-01-05
2024-01-08
2024-01-09
2024-01-10
2024-01-11
2024-01-12
2024-01-15
2024-01-16
2024-01-17
2024-01-18
2024-01-19
2024-01-22
2024-01-23
2024-01-24
2024-01-25
2024-01-26
2024-01-29
2024-01-30
2024-01-31
2024-02-01
2024-02-02
2024-02-05
2024-02-06
2024-02-07
2024-02-08
2024-02-09
2024-02-12
2024-02-13
2024-02-14
2024-02-15
2024-02-16
2024-02-19
2024-02-20
2024-02-21
2024-02-22
2024-02-23
2024-02-26
2024-02-27
2024-02-28
2024-02-29
2024-03-01
2024-03-04
2024-03-05
2024-03-06
2024-03-07
2024-03-08
2024-03-11
2024-03-12
2024-03-13
2024-03-14
2024-03-15
2024-03-18
2024-03-19
2024-03-20
2024-03-21
2024-03-22
2024-03-25
2024-03-26
2024-03-27
2024-03-28
2024-04-02
2024-04-03


In [ ]:
df_filtered.to_csv(r's:\Algo\Files\andrej\Data\int_data_lag_dem2_jan_aug_enriched_2leads_dem1_euadec1_burnout60_tp2_sl1_bamax03.csv')

In [ ]:
%store df_filtered